# Notebook 02 - Analisis Fiscal Argentina 2020-2026

**Fuente:** Secretaria de Hacienda | AIF (Sector Publico Base Caja) + IMIG  
**Deflactor:** IPC Nivel General INDEC  
**Unidad:** pesos constantes de abril 2026 (salvo indicacion contraria)

**Alcance de los datos:**
- **Titulares y ajuste:** *Sector Publico Total* = Administracion Nacional + PAMI + Fondos Fiduciarios  
  (coincide con lo que reporta Hacienda en sus comunicados)
- **Desagregacion por componentes:** solo Administracion Nacional (donde ocurrio el grueso del ajuste)

### Indice
1. Carga de datos y deflactor
2. Resultado Primario y Financiero 2020-2026
3. El ajuste fiscal 2024-2026: cuanto y de donde
4. Cuanto del ajuste se traslado a las provincias
5. Composicion del gasto (Administracion Nacional)
6. Composicion de ingresos (Administracion Nacional)
7. Desagregacion por subsector institucional
8. Exportar resultados a Excel

In [ ]:
# ── Celda 1: Dependencias ─────────────────────────────────────────────
import sys, io, requests
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q pandas matplotlib openpyxl

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (15, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
print('OK')

In [ ]:
# ── Celda 2: Carga de datos y funciones auxiliares ────────────────────
REPO     = 'https://raw.githubusercontent.com/santiagoriverti/cuentas_publicas/main'
REPO_BIN = 'https://github.com/santiagoriverti/cuentas_publicas/raw/main'

df_aif  = pd.read_csv(f'{REPO}/output/aif_consolidado.csv',  parse_dates=['fecha'])
df_imig = pd.read_csv(f'{REPO}/output/imig_consolidado.csv', parse_dates=['fecha'])
df      = df_aif[df_aif['periodo'] == 'mensual'].copy()

# IPC Nivel General
ipc_raw = pd.read_excel(
    io.BytesIO(requests.get(f'{REPO_BIN}/data/reference/IPC.xlsx').content),
    usecols=['date', 'Nivel general'])
ipc_raw['date'] = pd.to_datetime(ipc_raw['date'])
ipc = ipc_raw.set_index('date')['Nivel general'].sort_index()
ipc.index = ipc.index.to_period('M').to_timestamp()

BASE     = ipc.index.max()   # Abril 2026
IPC_BASE = ipc.loc[BASE]

# ── Funciones ─────────────────────────────────────────────────────────

def deflactar(serie):
    """Serie nominal -> pesos constantes de BASE (abril 2026)."""
    idx = serie.index.to_period('M').to_timestamp()
    return serie.values * (IPC_BASE / ipc.reindex(idx).values)

def get_serie(concepto, subsector='total_adm_nacional'):
    """Serie mensual para un concepto y subsector. Default: Administracion Nacional."""
    mask = (df['concepto_codigo'] == concepto) & (df['subsector'] == subsector)
    return df[mask].set_index('fecha')['valor_millones_pesos'].sort_index()

def get_serie_total(concepto):
    """
    Serie mensual del SECTOR PUBLICO TOTAL:
    Administracion Nacional + PAMI + Fondos Fiduciarios.
    Equivale al perimetro que usa Hacienda en sus comunicados.
    Para 2026 usa directamente total_general cuando esta disponible.
    """
    s_nac  = get_serie(concepto, 'total_adm_nacional')
    s_pami = get_serie(concepto, 'pami_fdos_otros')
    s_gen  = get_serie(concepto, 'total_general')   # disponible desde 2026

    # Combinar: usar total_general donde exista, sino sumar nac + pami
    s_suma = s_nac.add(s_pami.reindex(s_nac.index).fillna(0))
    # Donde existe total_general, reemplazar (mas preciso)
    if len(s_gen) > 0:
        s_suma.update(s_gen)
    return s_suma.sort_index()

print(f'AIF mensual  : {len(df):,} registros | {df.fecha.min().strftime("%Y-%m")} - {df.fecha.max().strftime("%Y-%m")}')
print(f'IPC          : {ipc.index.min().strftime("%Y-%m")} - {ipc.index.max().strftime("%Y-%m")}')
print(f'Base deflac. : {BASE.strftime("%Y-%m")} (IPC = {IPC_BASE:,.0f})')
print()
print('Alcance:')
print('  get_serie_total() -> Sector Publico Total (Adm.Nac + PAMI + Fondos Fiduciarios)')
print('  get_serie()       -> Solo Administracion Nacional')

## 1. Resultado Primario y Financiero 2020-2026
*Sector Publico Total — coincide con reportes oficiales de Hacienda.*

In [ ]:
primario   = get_serie_total('XIV_RESULTADO_PRIMARIO')
financiero = get_serie_total('XV_RESULTADO_FINANCIERO')
intereses  = get_serie_total('II2_INTERESES')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, vals_p, vals_f, titulo, ylabel in [
    (axes[0],
     primario.values / 1e6,
     financiero.values / 1e6,
     'Nominal (billones ARS corrientes)',
     'Billones ARS'),
    (axes[1],
     deflactar(primario) / 1e6,
     deflactar(financiero) / 1e6,
     f'Real (billones ARS de {BASE.strftime("%b %Y")})',
     'Billones ARS constantes'),
]:
    colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in vals_p]
    ax.bar(primario.index, vals_p, width=25, color=colors, alpha=0.85, label='Resultado Primario')
    ax.plot(financiero.index, vals_f, 'o-', color='#2c3e50', lw=2, ms=3, label='Resultado Financiero')
    ax.axhline(0, color='black', lw=0.8)
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
    ax.set_title(titulo, fontsize=11)
    ax.set_ylabel(ylabel)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Resultado Primario y Financiero - Sector Publico Total (mensual)', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Tabla anual
df_anual = pd.DataFrame({
    'Primario nominal (B ARS)':   primario   / 1e6,
    'Financiero nominal (B ARS)': financiero / 1e6,
    'Primario real (B ARS)':      deflactar(primario)   / 1e6,
    'Financiero real (B ARS)':    deflactar(financiero) / 1e6,
})
df_anual = df_anual.groupby(df_anual.index.year).sum()
print(f'\nResumen anual - Sector Publico Total (billones ARS, real = precios {BASE.strftime("%b %Y")}):')
print(df_anual.round(1).to_string())

## 2. El ajuste fiscal 2024-2026: cuanto y de donde
*Sector Publico Total para los titulares, Administracion Nacional para el desglose de componentes.*

In [ ]:
# Titulares con sector publico total
for concepto, label in [
    ('XIV_RESULTADO_PRIMARIO',  'Resultado Primario'),
    ('XV_RESULTADO_FINANCIERO', 'Resultado Financiero'),
    ('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT', 'Gasto Primario'),
]:
    s = get_serie_total(concepto)
    s_real = pd.Series(deflactar(s), index=s.index)
    anual = s_real.groupby(s_real.index.year).sum() / 1e6
    var_24 = anual.get(2024, float('nan')) - anual.get(2023, float('nan'))
    var_25 = anual.get(2025, float('nan')) - anual.get(2023, float('nan'))
    print(f'{label} (real, billones ARS):')
    print(f'  2022={anual.get(2022,0):.1f}  2023={anual.get(2023,0):.1f}  2024={anual.get(2024,0):.1f}  2025={anual.get(2025,0):.1f}')
    print(f'  Var 2023->2024: {var_24:+.1f} B   |   Var 2023->2025: {var_25:+.1f} B')
    print()

print('─' * 65)
# Desglose de componentes (Administracion Nacional)
print('\nDesglose por componente - Administracion Nacional (real, billones ARS):')
conceptos_ajuste = {
    'I_INGRESOS_CORRIENTES':        'Ingresos corrientes',
    'II_GASTOS_CORRIENTES':         'Gastos corrientes',
    'II2_INTERESES':                '  Intereses',
    'II3_PRESTACIONES_SEG_SOCIAL':  '  Prestaciones Seg.Social',
    'II1a_REMUNERACIONES':          '  Remuneraciones (salarios)',
    'II4b1_TRANSF_PROVINCIAS_CABA': '  Transf. a Provincias/CABA',
    'II4b2_TRANSF_UNIVERSIDADES':   '  Universidades',
    'II4a_TRANSF_SECTOR_PRIVADO':   '  Transf. Sector Privado (subsidios)',
    'V_GASTOS_CAPITAL':             'Gastos de capital',
    'XIV_RESULTADO_PRIMARIO':       'RESULTADO PRIMARIO',
}
rows = []
for cod, nombre in conceptos_ajuste.items():
    s = get_serie(cod)   # Administracion Nacional
    s_real = pd.Series(deflactar(s), index=s.index)
    for yr in [2022, 2023, 2024, 2025]:
        rows.append({'Concepto': nombre, 'Anio': yr,
                     'Real': s_real[s_real.index.year == yr].sum()})

df_ajuste = pd.DataFrame(rows).pivot(index='Concepto', columns='Anio', values='Real') / 1e6
df_ajuste['Var 23->24'] = df_ajuste[2024] - df_ajuste[2023]
df_ajuste['Var 23->25'] = df_ajuste[2025] - df_ajuste[2023]
df_ajuste['% ajuste 23->25'] = (df_ajuste['Var 23->25'] / abs(df_ajuste.loc['Gastos corrientes','Var 23->25'] + df_ajuste.loc['Gastos de capital','Var 23->25']) * 100).round(1)
df_ajuste = df_ajuste.reindex(list(conceptos_ajuste.values()))
print(df_ajuste.round(1).to_string())

# Grafico variacion 2023->2025
mask_detalle = df_ajuste.index.str.startswith('  ')
vals_plot = df_ajuste.loc[mask_detalle, 'Var 23->25'].dropna().sort_values()
fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#27ae60' if v >= 0 else '#e74c3c' for v in vals_plot]
bars = ax.barh(vals_plot.index.str.strip(), vals_plot.values, color=colors, alpha=0.85)
ax.axvline(0, color='black', lw=0.8)
ax.set_title('Variacion real por componente del gasto 2023 -> 2025 (Adm. Nacional)', fontsize=12)
ax.set_xlabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
for bar, val in zip(bars, vals_plot.values):
    ax.text(val + (0.3 if val >= 0 else -0.3), bar.get_y() + bar.get_height()/2,
            f'{val:+.1f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Cuanto del ajuste se traslado a las provincias
*Sector Publico Total para el denominador (gasto primario), Adm. Nacional para las transferencias.*

In [ ]:
# Transferencias corrientes + capital a Provincias/CABA
# (usamos Adm. Nacional: PAMI no transfiere directamente a provincias)
corr = get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'total_adm_nacional')
cap  = get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'total_adm_nacional')
total_prov = corr.add(cap.reindex(corr.index).fillna(0))

corr_tes = get_serie('II4b1_TRANSF_PROVINCIAS_CABA', 'tesoro_nacional')
cap_tes  = get_serie('V2a_TRANSF_CAPITAL_PROVINCIAS', 'tesoro_nacional')

# Deflactar
total_prov_real = pd.Series(deflactar(total_prov), index=total_prov.index)
corr_real = pd.Series(deflactar(corr_tes), index=corr_tes.index)
cap_real  = pd.Series(deflactar(cap_tes),  index=cap_tes.index)

# Grafico mensual
fig, ax = plt.subplots(figsize=(15, 5))
ax.stackplot(corr_real.index,
             corr_real.values / 1000,
             cap_real.reindex(corr_real.index).fillna(0).values / 1000,
             labels=['Corrientes (Tesoro)', 'Capital (Tesoro)'],
             colors=['#9b59b6', '#e67e22'], alpha=0.85)
ax.set_title('Transferencias Tesoro Nacional -> Provincias y CABA (pesos constantes)', fontsize=12)
ax.set_ylabel(f'Miles de MM ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

# Tabla y cuantificacion
gasto_total = get_serie_total('XII_GASTOS_PRIMARIOS_DESPUES_FIGURAT')
gasto_real  = pd.Series(deflactar(gasto_total), index=gasto_total.index)

resumen = pd.DataFrame({
    'Transf. prov. (B real)':  total_prov_real.groupby(total_prov_real.index.year).sum() / 1e6,
    'Gasto primario total (B)': gasto_real.groupby(gasto_real.index.year).sum() / 1e6,
})
resumen['Prov./Gasto (%)'] = (resumen['Transf. prov. (B real)'] / resumen['Gasto primario total (B)'] * 100)

var_prov  = resumen.loc[2024, 'Transf. prov. (B real)'] - resumen.loc[2023, 'Transf. prov. (B real)']
var_prov25= resumen.loc[2025, 'Transf. prov. (B real)'] - resumen.loc[2023, 'Transf. prov. (B real)']
var_gasto = resumen.loc[2024, 'Gasto primario total (B)'] - resumen.loc[2023, 'Gasto primario total (B)']
var_gasto25=resumen.loc[2025, 'Gasto primario total (B)'] - resumen.loc[2023, 'Gasto primario total (B)']

print('Transferencias a Provincias/CABA (Adm. Nacional) vs Gasto Primario Total (Sector Publico):')
print(resumen.round(1).to_string())
print()
print('=== CUANTIFICACION DEL AJUSTE A PROVINCIAS ===')
print(f'Var. transf. 2023->2024 : {var_prov:+.1f} B ARS  ({var_prov/resumen.loc[2023,"Transf. prov. (B real)"]*100:.0f}%)')
print(f'Var. transf. 2023->2025 : {var_prov25:+.1f} B ARS  ({var_prov25/resumen.loc[2023,"Transf. prov. (B real)"]*100:.0f}%)')
print(f'Var. gasto primario 23->24: {var_gasto:+.1f} B')
print(f'Var. gasto primario 23->25: {var_gasto25:+.1f} B')
print(f'Provincias / ajuste gasto 2023->2024: {var_prov/var_gasto*100:.1f}%')
print(f'Provincias / ajuste gasto 2023->2025: {var_prov25/var_gasto25*100:.1f}%')

## 4. Composicion del gasto corriente
*Administracion Nacional — donde se puede desagregar por tipo de gasto.*

In [ ]:
comp_gasto = {
    'II3_PRESTACIONES_SEG_SOCIAL': 'Prestaciones Seg.Social',
    'II2_INTERESES':               'Intereses deuda',
    'II1a_REMUNERACIONES':         'Remuneraciones',
    'II4a_TRANSF_SECTOR_PRIVADO':  'Transf. Sector Privado (subsidios)',
    'II4b1_TRANSF_PROVINCIAS_CABA':'Transf. Provincias/CABA',
    'II4b2_TRANSF_UNIVERSIDADES':  'Universidades',
}
colors_comp = ['#e74c3c','#8e44ad','#3498db','#2ecc71','#9b59b6','#f39c12']

idx = get_serie('II_GASTOS_CORRIENTES').index
fig, ax = plt.subplots(figsize=(15, 6))
bottom = np.zeros(len(idx))

for (cod, label), color in zip(comp_gasto.items(), colors_comp):
    vals = deflactar(get_serie(cod).reindex(idx).fillna(0)) / 1e6
    ax.bar(idx, vals, width=25, bottom=bottom, label=label, color=color, alpha=0.85)
    bottom += vals

ax.plot(idx, deflactar(get_serie('II_GASTOS_CORRIENTES')) / 1e6,
        'k-', lw=1.5, label='Total gastos corrientes')

ax.set_title('Composicion del Gasto Corriente - Administracion Nacional (real)', fontsize=12)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(loc='upper left', fontsize=8, ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Composicion de ingresos
*Administracion Nacional.*

In [ ]:
idx = get_serie('I_INGRESOS_CORRIENTES').index
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for ax, use_real, titulo in [
    (axes[0], False, 'Nominal (miles de MM ARS)'),
    (axes[1], True,  f'Real (billones ARS {BASE.strftime("%b %Y")})')
]:
    factor = 1e6 if use_real else 1e3
    bottom = np.zeros(len(idx))
    for cod, label, color in [
        ('I1_INGRESOS_TRIBUTARIOS',    'Tributarios',    '#2ecc71'),
        ('I2_APORTES_SEG_SOCIAL',      'Seg. Social',    '#3498db'),
        ('I3_INGRESOS_NO_TRIBUTARIOS', 'No tributarios', '#f39c12'),
    ]:
        s = get_serie(cod).reindex(idx).fillna(0)
        vals = (deflactar(s) if use_real else s.values) / factor
        ax.bar(idx, vals, width=25, bottom=bottom, label=label, color=color, alpha=0.85)
        bottom += vals
    s_tot = get_serie('I_INGRESOS_CORRIENTES')
    tot = (deflactar(s_tot) if use_real else s_tot.values) / factor
    ax.plot(idx, tot, 'k-', lw=1.5, label='Total')
    ax.set_title(titulo, fontsize=10)
    ax.set_ylabel('Billones ARS' if use_real else 'Miles de MM ARS')
    ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

plt.suptitle('Ingresos Corrientes - Administracion Nacional mensual', fontsize=12, y=1.02)
plt.tight_layout()
plt.show()

## 6. Desagregacion por subsector institucional
*Resultado Primario de cada subsector dentro de la Administracion Nacional.*

In [ ]:
subsectores = {
    'tesoro_nacional':      'Tesoro Nacional',
    'inst_seg_social':      'Seg. Social (ANSES)',
    'org_descentralizados': 'Org. Descentralizados',
    'rec_afectados':        'Rec. Afectados',
    'pami_fdos_otros':      'PAMI + Fondos Fiduciarios',
}
colors_ss = ['#e74c3c','#2ecc71','#3498db','#f39c12','#95a5a6']

fig, ax = plt.subplots(figsize=(15, 6))
for (ss, label), color in zip(subsectores.items(), colors_ss):
    s = get_serie('XIV_RESULTADO_PRIMARIO', ss)
    if len(s) == 0: continue
    ax.plot(s.index, deflactar(s) / 1e6, label=label, lw=1.8, color=color)

# Linea total sector publico
s_tot = get_serie_total('XIV_RESULTADO_PRIMARIO')
ax.plot(s_tot.index, deflactar(s_tot) / 1e6, 'k--', lw=2, label='Total Sector Publico')

ax.axhline(0, color='black', lw=0.8, ls=':')
ax.set_title('Resultado Primario por Subsector (real)', fontsize=12)
ax.set_ylabel(f'Billones ARS ({BASE.strftime("%b %Y")})')
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f'{x:,.1f}'))
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 7. Exportar resultados a Excel

In [ ]:
OUTPUT = 'analisis_fiscal_resultados.xlsx'

# ── Hoja 1: Serie mensual KPIs (Sector Publico Total) ────────────────
idx_base = get_serie_total('XIV_RESULTADO_PRIMARIO').index
rows_m = {'fecha': idx_base}
for nombre, cod in [
    ('ingresos_corrientes',  'I_INGRESOS_CORRIENTES'),
    ('gastos_corrientes',    'II_GASTOS_CORRIENTES'),
    ('intereses',            'II2_INTERESES'),
    ('prestaciones',         'II3_PRESTACIONES_SEG_SOCIAL'),
    ('resultado_primario',   'XIV_RESULTADO_PRIMARIO'),
    ('resultado_financiero', 'XV_RESULTADO_FINANCIERO'),
]:
    s = get_serie_total(cod)
    rows_m[f'{nombre}_nominal_MM_ARS'] = s.reindex(idx_base).values
    rows_m[f'{nombre}_real_MM_ARS']    = deflactar(s.reindex(idx_base))
df_serie = pd.DataFrame(rows_m)

# ── Hoja 2: Resumen anual ─────────────────────────────────────────────
df_anual_exp = df_serie.copy()
df_anual_exp['anio'] = pd.to_datetime(df_anual_exp['fecha']).dt.year
df_anual_exp = df_anual_exp.groupby('anio').sum(numeric_only=True)

# ── Hoja 3: Transferencias a provincias ──────────────────────────────
prov_rows = []
for cod, tipo in [('II4b1_TRANSF_PROVINCIAS_CABA','Corrientes'),
                  ('V2a_TRANSF_CAPITAL_PROVINCIAS','Capital')]:
    for ss in ['tesoro_nacional','total_adm_nacional']:
        s = get_serie(cod, ss)
        if len(s) == 0: continue
        prov_rows.append(pd.DataFrame({
            'fecha': s.index, 'tipo': tipo, 'subsector': ss,
            'nominal_MM_ARS': s.values,
            'real_MM_ARS':    deflactar(s),
        }))
df_prov_exp = pd.concat(prov_rows).sort_values(['fecha','tipo'])

# ── Hoja 4: Ajuste por componentes ───────────────────────────────────
df_ajuste_exp = df_ajuste.reset_index()

# ── Exportar ─────────────────────────────────────────────────────────
with pd.ExcelWriter(OUTPUT, engine='openpyxl') as writer:
    df_serie.to_excel(writer,        sheet_name='Serie_mensual',       index=False)
    df_anual_exp.to_excel(writer,    sheet_name='Resumen_anual',       index=True)
    df_prov_exp.to_excel(writer,     sheet_name='Transferencias_prov', index=False)
    df_ajuste_exp.to_excel(writer,   sheet_name='Ajuste_componentes',  index=False)

print(f'Exportado: {OUTPUT}')
print(f'Base deflacion: pesos de {BASE.strftime("%B %Y")}')
print('Hojas:')
print('  Serie_mensual       - KPIs mensuales nominal + real (Sector Publico Total)')
print('  Resumen_anual       - Totales anuales')
print('  Transferencias_prov - Corrientes + Capital a Provincias/CABA')
print('  Ajuste_componentes  - Variacion real por componente del gasto 2022-2025')

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT)
    print('Descarga iniciada')